# SeizeIT2 Dataset Feasibility Check

This notebook provides a preliminary inventory of the SeizeIT2 dataset for a project investigating non-convulsive focal seizure detection using non-EEG/non-ECG signals.

The main result is a compact summary of participant coverage, movement-file availability (a preliminary proxy for accelerometer and gyroscope data), EMG availability, and seizure-related annotations. These results help assess whether the dataset appears suitable for the proposed analysis.

**Interpretation limits:** file availability does not confirm that required sensor channels are present and usable, and seizure-related annotation labels do not establish the number of eligible non-convulsive focal seizures. These questions require further validation during the final project.

Dataset: [SeizeIT2, OpenNeuro ds005873](https://doi.org/10.18112/openneuro.ds005873.v1.1.0). The file inventory is retrieved from a public GitHub mirror, which may not exactly match the cited release. Counts should therefore be treated as preliminary.


## 1. Setup

The notebook retrieves public metadata and organizes it using pandas. An internet connection is required.


In [ ]:
import json
import urllib.request
from io import StringIO
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

BASE = "https://data.nemar.org/on005873/v1.0.0"
GITHUB_RAW = "https://raw.githubusercontent.com/OpenNeuroDatasets/ds005873/main/"
TREE_URL = "https://api.github.com/repos/OpenNeuroDatasets/ds005873/git/trees/main?recursive=1"
OUTPUT_DIR = Path("seizeit2_metadata_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Summary figures currently used in the project report. These are comparison targets, not forced results.
REPORT_EXPECTED = {
    "participants": 125,
    "recordings": 2850,
    "participants_with_movement": 117,
    "participants_with_ecg": 124,
    "participants_with_emg": 124,
    "male_participants": 63,
    "female_participants": 62,
}

def get_text(url, timeout=60):
    request = urllib.request.Request(
        url, headers={"User-Agent": "SeizeIT2-metadata-summary/1.0"}
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read().decode("utf-8")

def try_get_text(url, timeout=60):
    try:
        return get_text(url, timeout)
    except Exception as exc:
        print(f"Could not retrieve {url}: {exc}")
        return None

def normalize_participant_id(value):
    if pd.isna(value) or not str(value).strip():
        return pd.NA
    value = str(value).strip()
    return value if value.startswith("sub-") else f"sub-{value}"

def find_column(df, candidates):
    lookup = {str(c).lower(): c for c in df.columns}
    return next((lookup[c.lower()] for c in candidates if c.lower() in lookup), None)

print("Output directory:", OUTPUT_DIR.resolve())
print("Dataset metadata workflow initialized.")

## 2. Load participant metadata

Retrieve the participant-level metadata from the public dataset mirror.


In [ ]:
participants_text = try_get_text(f"{BASE}/participants.tsv")
participants_json_text = try_get_text(f"{BASE}/participants.json")
dataset_json_text = try_get_text(f"{BASE}/dataset_description.json")

if participants_text is None:
    raise RuntimeError("Could not retrieve participants.tsv. Check internet access and the NEMAR URL.")

participants_raw = pd.read_csv(StringIO(participants_text), sep="\t", dtype=str)
participants_dictionary = json.loads(participants_json_text) if participants_json_text else {}
dataset_description = json.loads(dataset_json_text) if dataset_json_text else {}

print("Dataset:", dataset_description.get("Name", "Not provided"))
print("Rows in participants.tsv:", len(participants_raw))
print("Columns:", participants_raw.columns.tolist())
print("Intermediate metadata prepared.")


## 3. Retrieve the file inventory

The file index provides paths and filenames, rather than the contents of the physiological recordings. Counts are preliminary until checked against the cited dataset release.


In [ ]:
tree_text = try_get_text(TREE_URL, timeout=90)
if tree_text is None:
    raise RuntimeError("Could not retrieve the public file index. Check your internet connection.")
tree_data = json.loads(tree_text)
if tree_data.get("truncated"):
    print("WARNING: GitHub file index is truncated; counts may be incomplete.")

file_paths_raw = [
    item["path"] for item in tree_data.get("tree", [])
    if item.get("type") == "blob"
]
print("Indexed paths:", len(file_paths_raw))
display(pd.DataFrame({"path": file_paths_raw[:20]}))

## 4. Organize participant and file information

Retain original paths and filenames while extracting participant IDs and modality folders. Participants are not removed when a modality is unavailable.


In [ ]:
# Keep the source modality folder name and identify common folder-name variants.
modality_folder_names = {
    "eeg", "ieeg", "ecg", "emg", "mov", "motion", "movement", "imu",
    "acc", "gyro", "accelerometer", "gyroscope", "beh"
}

inventory_rows = []
for path in file_paths_raw:
    parts = path.split("/")
    filename = parts[-1]
    participant_id = next((p for p in parts if p.startswith("sub-")), None)
    modality = next((p for p in parts if p.lower() in modality_folder_names), None)
    inventory_rows.append({
        "source_path_raw": path,
        "participant_id_raw": participant_id,
        "modality_folder_raw": modality,
        "filename_raw": filename,
        "extension_raw": "".join(Path(filename).suffixes),
        "is_edf_raw": filename.lower().endswith(".edf"),
        "is_events_tsv_raw": filename.lower().endswith("_events.tsv"),
        "is_scans_tsv_raw": filename.lower().endswith("_scans.tsv")
    })

file_inventory_raw = pd.DataFrame(inventory_rows)
print("File inventory rows:", len(file_inventory_raw))
print("Intermediate metadata prepared.")


## 5. Retrieve event annotations

Count available event annotation files from the file index. Full event-label validation is deferred to the final analysis.


In [ ]:
# For the midterm feasibility check, count event annotation files from the file index.
# Do not download every event TSV individually; detailed event-label validation is planned for the final analysis.
event_paths_raw = [p for p in file_paths_raw if p.lower().endswith("_events.tsv")]

event_inventory_rows = []
for path in event_paths_raw:
    participant_id = next((part for part in path.split("/") if part.startswith("sub-")), None)
    event_inventory_rows.append({
        "participant_id_from_path_raw": participant_id,
        "source_path_raw": path
    })

event_inventory_raw = pd.DataFrame(event_inventory_rows)
events_raw = pd.DataFrame()  # Full event contents are intentionally not downloaded at this stage.
event_failures = []

print("Event annotation files indexed:", len(event_paths_raw))
print("Participants represented in event-file paths:",
      event_inventory_raw["participant_id_from_path_raw"].nunique() if not event_inventory_raw.empty else 0)
print("Full annotation files downloaded: 0 (reserved for detailed validation)")


## 6. Preserve the source metadata

The source metadata and file inventory are saved before standardization so the original values can be reviewed if needed.


In [ ]:
participants_raw.to_csv(OUTPUT_DIR / "01_participants_uncleaned.tsv", sep="\t", index=False)
file_inventory_raw.to_csv(OUTPUT_DIR / "01_file_inventory_uncleaned.csv", index=False)
events_raw.to_csv(OUTPUT_DIR / "01_event_annotations_uncleaned.csv", index=False)
pd.DataFrame(event_failures, columns=["source_path", "error"]).to_csv(
    OUTPUT_DIR / "01_event_download_errors.csv", index=False
)

print("Uncleaned outputs saved in:", OUTPUT_DIR.resolve())
print("Intermediate metadata prepared.")
print("Intermediate metadata prepared.")
print("Intermediate metadata prepared.")


## 7. Standardize metadata

Standardize participant identifiers and modality labels while retaining source paths. Keep participants even when a modality is unavailable.


In [ ]:
file_inventory_clean = file_inventory_raw.copy()
file_inventory_clean["participant_id"] = file_inventory_clean["participant_id_raw"].apply(normalize_participant_id)

# Normalize common folder-name variants for counting while preserving modality_folder_raw.
modality_aliases = {
    "ieeg": "eeg",
    "motion": "mov",
    "movement": "mov",
    "imu": "mov",
    "acc": "mov",
    "gyro": "mov",
    "accelerometer": "mov",
    "gyroscope": "mov",
}
file_inventory_clean["modality"] = (
    file_inventory_clean["modality_folder_raw"]
    .fillna("participant-level").astype(str).str.strip().str.lower()
    .replace(modality_aliases)
)
file_inventory_clean["filename"] = file_inventory_clean["filename_raw"].astype("string").str.strip()
file_inventory_clean["is_signal_edf"] = file_inventory_clean["filename"].str.lower().str.endswith(".edf")
file_inventory_clean["is_event_annotation"] = file_inventory_clean["filename"].str.lower().str.endswith("_events.tsv")
file_inventory_clean["is_scans_metadata"] = file_inventory_clean["filename"].str.lower().str.endswith("_scans.tsv")

print("Normalized modality folders:")
print("Intermediate metadata prepared.")
print("Intermediate metadata prepared.")


## 8. Inspect event annotations

Event annotation files are counted from the file index in this preliminary workflow. Their contents and seizure labels will be validated during the final analysis.


In [ ]:
# Full event annotations are not downloaded in this preliminary inventory workflow.
# Keep an empty standardized table so later export cells remain consistent.
events_clean = pd.DataFrame()
print("Detailed event-label cleaning deferred to the final analysis.")


## 9. Create participant-level metadata

Combine participant details with file-availability counts to support the preliminary dataset summary.


In [ ]:
participants_clean = participants_raw.copy()
participant_id_col = find_column(participants_clean, ["participant_id", "participant", "subject_id", "sub"])
if participant_id_col is None:
    raise ValueError("Could not identify participant ID column in participants.tsv.")

participants_clean["participant_id"] = participants_clean[participant_id_col].apply(normalize_participant_id)
for col in participants_clean.columns:
    if col != "participant_id":
        values = participants_clean[col].astype("string").str.strip()
        participants_clean[col] = values.replace({"": pd.NA, "n/a": pd.NA, "NA": pd.NA, "nan": pd.NA})

# Count indexed files by participant and modality. No signal files are downloaded.
file_counts = (
    file_inventory_clean.dropna(subset=["participant_id"])
    .groupby("participant_id")
    .agg(
        total_indexed_files=("source_path_raw", "count"),
        signal_edf_files=("is_signal_edf", "sum"),
        event_annotation_files=("is_event_annotation", "sum"),
        scans_metadata_files=("is_scans_metadata", "sum")
    ).reset_index()
)

for modality in ["eeg", "ecg", "emg", "mov"]:
    modality_counts = (
        file_inventory_clean.loc[
            (file_inventory_clean["modality"] == modality) & file_inventory_clean["is_signal_edf"]
        ].groupby("participant_id").size().rename(f"{modality}_edf_files").reset_index()
    )
    file_counts = file_counts.merge(modality_counts, on="participant_id", how="left")

participant_summary_clean = participants_clean.merge(file_counts, on="participant_id", how="left")

count_cols = [
    c for c in participant_summary_clean.columns
    if c.endswith("_files") or c in {"total_indexed_files", "signal_edf_files"}
]
for col in count_cols:
    participant_summary_clean[col] = pd.to_numeric(
        participant_summary_clean[col], errors="coerce"
    ).fillna(0).astype(int)

for modality in ["eeg", "ecg", "emg", "mov"]:
    count_col = f"{modality}_edf_files"
    if count_col not in participant_summary_clean:
        participant_summary_clean[count_col] = 0
    participant_summary_clean[f"has_{modality}_edf"] = participant_summary_clean[count_col] > 0

# Annotation contents are intentionally not downloaded in this preliminary workflow.
participant_summary_clean["annotation_rows"] = pd.NA
participant_summary_clean["annotation_rows_with_seizure_label"] = pd.NA

age_cols = [
    c for c in participant_summary_clean.columns
    if c.lower() in {"age", "age_years", "participant_age", "age_at_recording"}
]
sex_cols = [
    c for c in participant_summary_clean.columns
    if c.lower() in {"sex", "gender"}
]
print("Age field(s) available:", age_cols if age_cols else "No explicit age field found")
print("Participant rows retained:", len(participant_summary_clean))

front = ["participant_id"] + age_cols + sex_cols
for col in [
    "eeg_edf_files", "ecg_edf_files", "emg_edf_files", "mov_edf_files",
    "signal_edf_files", "event_annotation_files"
]:
    if col in participant_summary_clean and col not in front:
        front.append(col)
participant_summary_clean = participant_summary_clean[
    front + [c for c in participant_summary_clean.columns if c not in front]
]
participant_summary_full_clean = participant_summary_clean.copy()
print("Participant-level metadata summary prepared.")


## 10. Create the main analysis table

The participant-level table focuses on movement-file availability (a preliminary proxy for accelerometer and gyroscope data), EMG availability, and seizure-related annotation information. File presence alone does not confirm usable channels or eligible seizure events.


In [ ]:
# Build the main analysis table from the full cleaned metadata.
participant_summary_full_clean["has_movement_and_emg_files"] = (
    participant_summary_full_clean["has_mov_edf"] & participant_summary_full_clean["has_emg_edf"]
)

# These flags describe file availability only. They do not establish usable channels or eligible seizures.
participant_summary_full_clean["movement_file_available_for_acc_gyr_screen"] = (
    participant_summary_full_clean["has_mov_edf"]
)
participant_summary_full_clean["movement_and_emg_files_available_for_screen"] = (
    participant_summary_full_clean["has_movement_and_emg_files"]
)

# Include participant details where available, non-EEG/non-ECG file indicators, and annotation summaries.
analysis_columns = ["participant_id"]
for col in ["age", "age_years", "participant_age", "age_at_recording", "sex", "gender"]:
    if col in participant_summary_full_clean.columns and col not in analysis_columns:
        analysis_columns.append(col)

for col in [
    "mov_edf_files", "has_mov_edf", "emg_edf_files", "has_emg_edf",
    "has_movement_and_emg_files", "event_annotation_files", "annotation_rows",
    "annotation_rows_with_seizure_label", "movement_file_available_for_acc_gyr_screen",
    "movement_and_emg_files_available_for_screen"
]:
    if col in participant_summary_full_clean.columns and col not in analysis_columns:
        analysis_columns.append(col)

participant_analysis_table_clean = participant_summary_full_clean[analysis_columns].copy()
participant_analysis_table_clean = participant_analysis_table_clean.rename(columns={
    "mov_edf_files": "movement_edf_files_proxy_for_acc_gyr",
    "has_mov_edf": "has_movement_files_proxy_for_acc_gyr",
})

print("Intermediate metadata prepared.")
print("Participants retained in main analysis table:", len(participant_analysis_table_clean))
print("Participants with movement files (ACC/GYR proxy):", int(participant_summary_full_clean["has_mov_edf"].sum()))
print("Participants with both movement and EMG files:", int(participant_summary_full_clean["has_movement_and_emg_files"].sum()))


## 11. Check summary counts

Compare selected counts currently used in the project report with counts calculated from the retrieved metadata. Differences should be investigated rather than overwritten, as they may reflect file definitions, folder names, or dataset version differences.


In [ ]:
computed = {
    "participants": int(participant_summary_full_clean["participant_id"].nunique()),
    "recordings": int(participant_summary_full_clean["signal_edf_files"].sum()) if "signal_edf_files" in participant_summary_full_clean else np.nan,
    "participants_with_movement": int(participant_summary_full_clean["has_mov_edf"].sum()),
    "participants_with_ecg": int(participant_summary_full_clean["has_ecg_edf"].sum()),
    "participants_with_emg": int(participant_summary_full_clean["has_emg_edf"].sum()),
}

checks = []
for metric, expected in REPORT_EXPECTED.items():
    actual = computed.get(metric, np.nan)
    if metric in {"male_participants", "female_participants"}:
        sex_col = next((c for c in participant_summary_full_clean.columns if c.lower() in {"sex", "gender"}), None)
        if sex_col:
            vals = participant_summary_full_clean[sex_col].astype("string").str.lower().str.strip()
            if metric == "male_participants":
                actual = int(vals.isin(["m", "male"]).sum())
            else:
                actual = int(vals.isin(["f", "female"]).sum())
        else:
            actual = "No sex/gender field found"
    checks.append({
        "metric": metric,
        "report_value": expected,
        "computed_from_current_inventory": actual,
        "match": (actual == expected) if isinstance(actual, (int, np.integer, float, np.floating)) and not pd.isna(actual) else "Needs review"
    })

report_consistency_check = pd.DataFrame(checks)
print("Report count comparison completed; details saved to 03_report_consistency_check.csv.")

## 12. Save the main outputs

Save the compact dataset summary and participant-level table for inspection. Other intermediate files are retained to support traceability.


In [ ]:
participants_clean.to_csv(OUTPUT_DIR / "02_participants_cleaned.tsv", sep="\t", index=False)
file_inventory_clean.to_csv(OUTPUT_DIR / "02_file_inventory_cleaned.csv", index=False)
events_clean.to_csv(OUTPUT_DIR / "02_event_annotations_cleaned.csv", index=False)
participant_summary_full_clean.to_csv(OUTPUT_DIR / "02_participant_metadata_full_cleaned_for_audit.csv", index=False)
participant_analysis_table_clean.to_csv(OUTPUT_DIR / "02_participant_analysis_table_cleaned.csv", index=False)
report_consistency_check.to_csv(OUTPUT_DIR / "03_report_consistency_check.csv", index=False)

print("Saved outputs in:", OUTPUT_DIR.resolve())
print("Main summary: 04_preliminary_dataset_summary.csv")
print("Participant table: 02_participant_analysis_table_cleaned.csv")


## Interpretation notes

- A missing modality flag means the current file index did not identify a matching file. It is not a reason to remove a participant from the metadata inventory.
- Movement-file availability is only a proxy for accelerometer and gyroscope availability until channel names are checked.
- Seizure-related annotation labels are a preliminary screen, not a validated count of non-convulsive focal seizures.
- Before model evaluation, recordings from the same participant should remain in the same data partition to reduce participant-level data leakage.


## Output files

The main results are `04_preliminary_dataset_summary.csv` (compact dataset counts) and `02_participant_analysis_table_cleaned.csv` (participant-level availability). Additional metadata files are retained for traceability. The event-annotation export is an empty placeholder in this preliminary workflow because annotation contents are not downloaded.


## 13. Preliminary dataset summary

Create a compact summary of participant coverage and non-EEG/non-ECG signal availability.


In [ ]:
# Summarize the retrieved participant and file inventory.
n_participants = int(participant_summary_full_clean["participant_id"].nunique())
n_movement = int(participant_summary_full_clean["has_mov_edf"].sum())
n_emg = int(participant_summary_full_clean["has_emg_edf"].sum())
n_movement_emg = int(participant_summary_full_clean["has_movement_and_emg_files"].sum())
n_signal_edf = int(participant_summary_full_clean["signal_edf_files"].sum())

dataset_summary_table = pd.DataFrame([
    {
        "Measure": "Participants represented in participant metadata",
        "Result": n_participants,
        "Interpretation": "Participant-level count from the retrieved metadata."
    },
    {
        "Measure": "Indexed signal EDF files",
        "Result": n_signal_edf,
        "Interpretation": "File-index count; may differ from the study's definition of a recording."
    },
    {
        "Measure": "Participants with movement EDF files",
        "Result": n_movement,
        "Interpretation": "Preliminary proxy for ACC/GYR availability; channel names require verification."
    },
    {
        "Measure": "Participants with EMG EDF files",
        "Result": n_emg,
        "Interpretation": "File presence does not confirm usable signal quality."
    },
    {
        "Measure": "Participants with both movement and EMG EDF files",
        "Result": n_movement_emg,
        "Interpretation": "Potential cohort before applying seizure, channel, and quality eligibility criteria."
    },
    {
        "Measure": "Usable annotated non-convulsive focal seizures with required signals",
        "Result": "Not yet established",
        "Interpretation": "Requires validation of event labels, timestamps, required channels, and usable recording duration."
    },
])

display(dataset_summary_table)
dataset_summary_table.to_csv(OUTPUT_DIR / "04_preliminary_dataset_summary.csv", index=False)
print("Saved dataset summary to:", (OUTPUT_DIR / "04_preliminary_dataset_summary.csv").resolve())


## 14. Preliminary interpretation

The interpretation below describes data availability and analysis feasibility. It does not estimate seizure-detection performance.


In [ ]:
print("Preliminary interpretation")
print()
print(
    f"The inventory currently represents {n_participants} participants and "
    f"{n_signal_edf} indexed signal EDF files. Movement-folder EDF files were identified "
    f"for {n_movement} participants, EMG EDF files for {n_emg} participants, and both "
    f"movement and EMG EDF files for {n_movement_emg} participants."
)
print()
print(
    "These counts describe the retrieved file inventory, not confirmed channel-level availability "
    "or usable data. The movement folder must be checked for accelerometer and gyroscope channels, "
    "and the event annotations must be reviewed to identify eligible non-convulsive focal seizures. "
    "The number of usable target seizures with the required signals is not yet established because event-label contents have not been downloaded and validated."
)
print()
print(
    "The inventory can support a preliminary assessment of data coverage and analysis feasibility, "
    "but it cannot establish seizure-detection performance. Before training models, verify the dataset "
    "release and channel names, validate event labels and timestamps, define recording and participant "
    "eligibility, preprocess and align the signals, and create a participant-separated evaluation plan. "
    "The planned ACC-only, ACC+GYR, and ACC+GYR+EMG comparisons should use the same eligible participants "
    "wherever possible."
)
print()
print(
    "After model development, performance can be assessed using seizure-level sensitivity, false alarms "
    "per 24 hours, and detection latency, with uncertainty estimates where the available data permit. "
    "Until that work is completed, no conclusion about detection accuracy or generalization to unseen "
    "participants is supported."
)


In [ ]:
print("Main outputs saved:")
print("- 04_preliminary_dataset_summary.csv")
print("- 02_participant_analysis_table_cleaned.csv")
